# 02: Embedding Extraction — Extract and Cache from All Models

**Goal**: Extract embeddings from SBERT and CLIP (text-only) on the VSR dataset and cache results.

This is the most computationally expensive step — results are cached to disk as .npy files.

All embeddings are L2-normalized before saving.

**Note**: This notebook can run locally on CPU or GPU. For faster extraction, use Google Colab with GPU acceleration.

In [ ]:
# Colab setup — skip this cell if running locally
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os
    import subprocess
    import getpass

    from google.colab import drive
    drive.mount("/content/drive")

    subprocess.run([
        "pip", "install", "-q",
        "torch", "torchvision", "transformers", "sentence-transformers",
        "datasets", "Pillow", "python-dotenv", "tqdm",
        "scikit-learn", "scipy", "pandas", "matplotlib", "seaborn",
    ], check=True)

    REPO_DIR = "/content/spatial-probing"
    if not os.path.exists(REPO_DIR):
        gh_token = getpass.getpass("GitHub Personal Access Token (ghp_...): ")
        clone_url = f"https://{gh_token}@github.com/nogully/spatial-probing.git"
        result = subprocess.run(
            ["git", "clone", clone_url, REPO_DIR],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"git clone failed:\n{result.stderr}")
        print("Clone succeeded.")
    else:
        print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    os.environ["CACHE_DIR"] = "/content/drive/MyDrive/spatial_probing/results/embeddings"
    os.environ["HF_TOKEN"] = ""  # paste your HuggingFace token here

    print("Colab setup complete.")
else:
    print("Running locally — skipping Colab setup.")

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
from dotenv import load_dotenv
import os

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

# Load environment
load_dotenv()
CACHE_DIR = Path(os.getenv("CACHE_DIR", "../results/embeddings"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

print(f"Cache directory: {CACHE_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 2. Load VSR Dataset

In [ ]:
from datasets import load_dataset

print("Loading VSR dataset...")
vsr = load_dataset("cambridgeltl/vsr_random")
train_data = vsr["train"]
print(f"VSR train set: {len(train_data)} examples")

# Extract texts and images
texts = [ex["caption"] for ex in train_data]
images = [ex["image"] for ex in train_data]

print(f"Extracted: {len(texts)} captions, {len(images)} images")
print(f"Sample captions:")
for i in range(3):
    print(f"  {i+1}. {texts[i]}")

## 3. Extract SBERT Embeddings

In [ ]:
from src.embedders import SBERTEmbedder

# Check cache
sbert_cache = CACHE_DIR / "sbert_vsr_train.npy"

if sbert_cache.exists():
    print(f"Loading cached SBERT embeddings from {sbert_cache}")
    sbert_embeddings = np.load(sbert_cache)
    print(f"Loaded shape: {sbert_embeddings.shape}")
else:
    print(f"Computing SBERT embeddings...")
    embedder = SBERTEmbedder()
    sbert_embeddings = embedder.embed_text(texts, batch_size=64)
    print(f"Extracted embeddings: {sbert_embeddings.shape}")
    
    # Verify L2 normalization
    norms = np.linalg.norm(sbert_embeddings, axis=1)
    print(f"Embedding norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")
    
    # Cache
    np.save(sbert_cache, sbert_embeddings.astype(np.float32))
    print(f"Saved to {sbert_cache}")

## 4. Extract CLIP Text Embeddings

In [ ]:
from src.embedders import CLIPTextEmbedder

# Check cache
clip_text_cache = CACHE_DIR / "clip_text_vsr_train.npy"

if clip_text_cache.exists():
    print(f"Loading cached CLIP text embeddings from {clip_text_cache}")
    clip_text_embeddings = np.load(clip_text_cache)
    print(f"Loaded shape: {clip_text_embeddings.shape}")
else:
    print(f"Computing CLIP text embeddings...")
    embedder = CLIPTextEmbedder()
    clip_text_embeddings = embedder.embed_text(texts, batch_size=32)
    print(f"Extracted embeddings: {clip_text_embeddings.shape}")
    
    # Verify L2 normalization
    norms = np.linalg.norm(clip_text_embeddings, axis=1)
    print(f"Embedding norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")
    
    # Cache
    np.save(clip_text_cache, clip_text_embeddings.astype(np.float32))
    print(f"Saved to {clip_text_cache}")

## 5. Extract CLIP Multimodal Embeddings (Image + Text)

In [ ]:
from src.embedders import CLIPMultimodalEmbedder

# Check cache
clip_mm_cache = CACHE_DIR / "clip_multimodal_vsr_train.npy"

if clip_mm_cache.exists():
    print(f"Loading cached CLIP multimodal embeddings from {clip_mm_cache}")
    clip_mm_embeddings = np.load(clip_mm_cache)
    print(f"Loaded shape: {clip_mm_embeddings.shape}")
else:
    print(f"Computing CLIP multimodal (image+text) embeddings...")
    embedder = CLIPMultimodalEmbedder()
    clip_mm_embeddings = embedder.embed_image_text(images, texts, batch_size=32)
    print(f"Extracted embeddings: {clip_mm_embeddings.shape}")
    
    # Verify L2 normalization
    norms = np.linalg.norm(clip_mm_embeddings, axis=1)
    print(f"Embedding norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")
    
    # Cache
    np.save(clip_mm_cache, clip_mm_embeddings.astype(np.float32))
    print(f"Saved to {clip_mm_cache}")

## 6. Sanity Checks

In [ ]:
# Verify all embeddings match
print("Embedding Shapes:")
print(f"  SBERT:        {sbert_embeddings.shape}")
print(f"  CLIP text:    {clip_text_embeddings.shape}")
print(f"  CLIP mm:      {clip_mm_embeddings.shape}")

# Check dimensions
assert sbert_embeddings.shape[0] == len(texts), "SBERT embedding count mismatch"
assert clip_text_embeddings.shape[0] == len(texts), "CLIP text embedding count mismatch"
assert clip_mm_embeddings.shape[0] == len(texts), "CLIP mm embedding count mismatch"

print(f"\nDimensions:")
print(f"  SBERT:   {sbert_embeddings.shape[1]}d")
print(f"  CLIP:    {clip_text_embeddings.shape[1]}d")

# Verify L2 normalization for all
print(f"\nL2 Normalization Check:")
for name, emb in [("SBERT", sbert_embeddings), ("CLIP text", clip_text_embeddings), ("CLIP mm", clip_mm_embeddings)]:
    norms = np.linalg.norm(emb, axis=1)
    print(f"  {name:12s}: norms in [{norms.min():.4f}, {norms.max():.4f}] (should be ~1.0)")

# Compute similarities to sanity check
print(f"\nCosine Similarity Sanity Check (first pair):")
for name, emb in [("SBERT", sbert_embeddings), ("CLIP text", clip_text_embeddings), ("CLIP mm", clip_mm_embeddings)]:
    sim = np.dot(emb[0], emb[1])  # Should be high (same sentence type)
    print(f"  {name:12s}: sim(emb[0], emb[1]) = {sim:.4f}")

## 7. Save Metadata

In [ ]:
import json
from datetime import datetime

# Save metadata about extraction
metadata = {
    "dataset": "vsr_random",
    "split": "train",
    "n_examples": len(texts),
    "extraction_date": datetime.now().isoformat(),
    "models": {
        "sbert": {
            "name": "sentence-transformers/all-mpnet-base-v2",
            "dim": int(sbert_embeddings.shape[1]),
            "file": "sbert_vsr_train.npy"
        },
        "clip_text": {
            "name": "openai/clip-vit-base-patch32",
            "type": "text-only",
            "dim": int(clip_text_embeddings.shape[1]),
            "file": "clip_text_vsr_train.npy"
        },
        "clip_multimodal": {
            "name": "openai/clip-vit-base-patch32",
            "type": "image+text",
            "dim": int(clip_mm_embeddings.shape[1]),
            "file": "clip_multimodal_vsr_train.npy"
        }
    },
    "normalization": "L2"
}

metadata_path = CACHE_DIR / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata to {metadata_path}")
print(f"\nMetadata:")
print(json.dumps(metadata, indent=2))

## 8. Summary

In [ ]:
print("\n" + "="*70)
print("EMBEDDING EXTRACTION COMPLETE")
print("="*70)
print(f"\nDataset: VSR (vsr_random, train split)")
print(f"Examples: {len(texts)}")
print(f"\nCached embeddings:")
print(f"  ✓ SBERT (768d):      {CACHE_DIR / 'sbert_vsr_train.npy'}")
print(f"  ✓ CLIP text (512d):  {CACHE_DIR / 'clip_text_vsr_train.npy'}")
print(f"  ✓ CLIP mm (512d):    {CACHE_DIR / 'clip_multimodal_vsr_train.npy'}")
print(f"\nFiles:")
for f in sorted(CACHE_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name:30s} {size_mb:8.1f} MB")
print(f"\nAll embeddings are L2-normalized (norms ~1.0)")
print(f"Ready for probing experiments!")
print("="*70)